# Ingesta de relaciones película-compañía

Lee la carpeta `movie_company` (CSV) desde Bronze, aplica un esquema explícito, estandariza las columnas y guarda el resultado como Parquet en `silver/movies_companies` (modo `overwrite`).

### Configuración

Carga las rutas de almacenamiento y las funciones comunes, y lee el parámetro `p_environment`.

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, lit, current_timestamp

### Lectura del archivo

Primero lee la carpeta `movie_company` con inferencia para inspeccionarlo y luego lo relee con un esquema explícito.

In [0]:
genre_df = (
    spark.read
        .format("csv")
        .load(f"{bronze_folder_path}/movie_company")
) 
display(genre_df)

In [0]:
movies_company_df_shema = StructType([
    StructField("movieId", IntegerType(), True),
    StructField("companyId", StringType(), True)
])
movies_company_df = (
    spark.read
        .format('csv')
        .schema(movies_company_df_shema)
        .load(f"{bronze_folder_path}/movie_company")
)
display(movies_company_df)
movies_company_df.printSchema()

### Transformación de columnas

Renombra `movieId` y `companyId` a snake_case y agrega `ingestion_date` y `env`.

In [0]:
movies_company_df = (
    add_ingestion_date(movies_company_df)
    .select(
        col("movieId").alias("movie_id"),
        col("companyId").alias("company_id"),
        lit(v_env).alias("env")
    )
)
display(movies_company_df)

### Escritura en Silver

Guarda el resultado como Parquet en `silver/movies_companies` en modo `overwrite`.

In [0]:
(
    movies_company_df.write
    .format("parquet")
    .mode("overwrite")
    .save(f"{silver_folder_path}/movies_companies")
)

### Fin

Devuelve `Success` al notebook orquestador.

In [0]:
dbutils.notebook.exit("Success")